# L10 · SOD와 SAGE-OPD

## Goal

**예상 시간:** 45분 · **경로:** full

- SOD step weight를 계산한다
- SAGE intervention을 조합한다
- loss scale을 보존한다

### 현재 위치: L09 → **L10** → L11

```text
Prompt/Data -> state source -> ... -> L10 -> ... -> fair evaluation
```

Alt text: The course map highlights L10 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L10"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L10', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

SOD는 step divergence로 신뢰하기 어려운 구간을 낮추고, SAGE-OPD는 intervention과 teacher confidence를 곱한 뒤 dense OPD와 loss scale을 맞춘다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

SOD는 step별 teacher/student divergence를 계산해 급격히 불일치하는 step의 distillation weight를 조절한다. weight를 detach할지 여부는 단순 구현 취향이 아니라 objective 의미를 바꾼다. 이 mini 구현은 안정적인 curriculum signal로 취급해 gradient를 끊는다.

SAGE-OPD는 teacher intervention 필요도와 teacher confidence를 곱해 token weight를 만들고, 평균 weight가 dense OPD와 같도록 정규화한다. intervention 0인 batch는 skip/fallback 정책이 필요하다. mini judge는 token-agreement proxy이며 semantic judge 성능을 주장하지 않는다.

### 실제 구현: 왜 이렇게 만들었나

SOD는 token KL을 step별 평균으로 모아 detached weight로 다시 token에 broadcast한다. SAGE는 intervention label, confidence, normalization을 각각 함수로 분리해 ablation할 수 있다. SOD 논문의 별도 GRPO 항은 구현하지 않았고 metric에 표시한다.

실제 코드: [`sod.py`](../../src/opd_study/algorithms/sod.py), [`sage_opd.py`](../../src/opd_study/algorithms/sage_opd.py).

In [2]:
import inspect
from opd_study.algorithms.sod import step_divergence_weights
from opd_study.algorithms.sage_opd import sage_opd_loss

objects_to_show = (step_divergence_weights, sage_opd_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.sod.step_divergence_weights
def step_divergence_weights(
    student_logits: Tensor,
    teacher_logits: Tensor,
    token_ids: Tensor,
    response_mask: Tensor,
    step_ids: Tensor,
    *,
    epsilon: float = 1e-6,
    delta: float = 0.2,
) -> tuple[Tensor, Tensor]:
    """Compute detached SOD ``d_k`` and cumulative-ratio ``w_k`` per token.

    ``d_k`` is the mean absolute sampled-token log-probability gap within a step.
    ``w_k = min(prod_{u<k}(d_u+eps)/(d_{u+1}+eps), 1+delta)``.
    """

    if student_logits.shape != teacher_logits.shape:
        raise ValueError("student and teacher logits must match")
    if token_ids.shape != response_mask.shape or token_ids.shape != step_ids.shape:
        raise ValueError("token_ids, response_mask and step_ids must match")
    if epsilon <= 0 or delta < 0:
        raise ValueError("epsilon must be positive and delta non-negative")
    student_log = torch.log_softmax(student_logits.float(), dim=-1)
    teacher_log 

### 다른 선택지는 없나?

SOD와 SAGE는 경쟁하는 단일 recipe라기보다 다른 실패 신호를 쓴다. step divergence가 신뢰도 proxy이면 SOD, recoverability/teacher 판단을 직접 물을 수 있으면 SAGE가 자연스럽다. 둘을 합치려면 weight scale과 double-counting을 새로 검증한다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L10의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.algorithms.sod import step_divergence_weights
from opd_study.data import CharacterTokenizer, collate_multiturn_text, generate_tiny_arithmetic

tokenizer = CharacterTokenizer(); rows = generate_tiny_arithmetic(train_rows=2, validation_rows=1, test_rows=1).train
batch = collate_multiturn_text([(row.prompt, tuple(row.response.splitlines())) for row in rows], tokenizer)
shape = (*batch.token_ids.shape, tokenizer.vocab_size)
student_logits, teacher_logits = torch.randn(shape), torch.randn(shape)
divergence, sod_weights = step_divergence_weights(student_logits, teacher_logits,
    batch.token_ids, batch.response_mask, batch.step_ids)
print("SOD mean divergence/weight:", float(divergence[divergence > 0].mean()),
      float(sod_weights[sod_weights > 0].mean()))

SOD mean divergence/weight: 1.0277701616287231 1.0374205112457275


In [4]:
from opd_study.algorithms.sage_opd import sage_token_weights

number_of_turns = int(batch.turn_ids.max()) + 1
intervention = torch.tensor([[0.0, 0.5, 1.0]]).repeat(batch.token_ids.shape[0], 1)
sage_weights, confidence = sage_token_weights(teacher_logits, batch, intervention)
print("SAGE normalized token-weight sum:", float(sage_weights.sum()))
print("response token count:", int(batch.response_mask[:, 1:].sum()))
print("turn confidence row 0:", confidence[0].tolist())

SAGE normalized token-weight sum: 69.99999237060547
response token count: 70
turn confidence row 0: [0.10730810463428497, 0.09619998186826706, 0.0]


## Checks

In [5]:
assert not divergence.requires_grad and not sod_weights.requires_grad
assert abs(float(sage_weights.sum()) - int(batch.response_mask[:, 1:].sum())) < 1e-4
assert (sage_weights >= 0).all()
print("check passed: SOD weights detach; SAGE preserves dense-OPD loss scale")

check passed: SOD weights detach; SAGE preserves dense-OPD loss scale


**연습 (10분):** SAGE intervention을 전부 0과 전부 1로 바꿔 skip/normalization 동작을 비교하고, SOD weight에 gradient가 없는지 재확인하라.

<details><summary>확인 기준</summary>빈 intervention은 명시적으로 처리되고, dense case의 weight 합은 response token 수에 맞으며 SOD weight는 detached다.</details>

## 내가 자주 틀리는 것

### M1 — SOD weight gradient 정책을 생략하기

- 틀린 형태: detach 여부를 프레임워크 기본값에 맡긴다.
- 왜 틀렸나: objective와 second-order 경로가 달라진다.
- 고친 형태: curriculum weight로 detach한다고 코드·문서·테스트에 고정한다.
- 관련 검사: `test_sod_downweights_a_divergence_jump`

### M2 — mini SAGE proxy를 semantic judge로 부르기

- 틀린 형태: token agreement proxy를 teacher 판단 성능으로 보고한다.
- 왜 틀렸나: 의미적 recoverability를 측정하지 않는다.
- 고친 형태: proxy label과 research judge 미검증 상태를 표시한다.
- 관련 검사: `test_sage_weights_normalize_and_skip`

## 60초 요약

1. SOD step weight를 계산한다
2. SAGE intervention을 조합한다
3. loss scale을 보존한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`sod`](https://arxiv.org/abs/2605.07725v3) · `2605.07725v3` · license `arXiv-non-exclusive-distribution-1.0` · [audited manifest](../../docs/sources.yml)
- [`sage_opd`](https://arxiv.org/abs/2606.19659v1) · `2606.19659v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`sod_official`](https://github.com/YoungZ365/SOD) · `110c4b8e843aee274d3cd648199569369ee2403e` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)